In [35]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
import os


In [36]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [37]:
llm = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)

In [38]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str


In [39]:
def generate_joke(state:JokeState):
    prompt=f'Generate a joke on the topic of {state['topic']}'
    response=llm.invoke(prompt).content
    
    return {'joke':response}

def explain_joke(state:JokeState):
    prompt=f'Generate a explaination on the given joke: {state['joke']}'
    response=llm.invoke(prompt).content
    
    return {'explaination':response}


In [40]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('explain_joke',explain_joke)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','explain_joke')
graph.add_edge('explain_joke',END)

checkpointer= MemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [41]:
thread_id=1

config={'configurable':{'thread_id':thread_id}}

intial_state={'topic':'burger'}

workflow.invoke(intial_state,config=config)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 16.34573398s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explaination': 'This joke is a classic example of a **pun**, which relies on a word having two different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Buns" (Meaning 1 - Food Context):** When you hear "Why did the *burger* go to the gym?", your mind immediately associates "buns" with the bread rolls that hold the burger patty. A good burger definitely needs "better buns" in this sense – fresh, soft, well-formed bread.\n\n2.  **"Buns" (Meaning 2 - Fitness Context):** In the context of going to the *gym*, "buns" is a common, informal term for the gluteal muscles (your buttocks). People go to the gym to "work on their buns" to make them stronger, firmer, or more shapely.\n\n**The Humor:**\n\nThe joke creates its humor by playing on this double meaning. You expect the punchline to relate to the burger\'s components, but it cleverly switches to the human/fitness meaning of "buns," c

In [ ]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explaination': 'This joke is a classic example of a **pun**, which relies on a word having two different meanings.\n\nHere\'s the breakdown:\n\n1.  **"Buns" (Meaning 1 - Food Context):** When you hear "Why did the *burger* go to the gym?", your mind immediately associates "buns" with the bread rolls that hold the burger patty. A good burger definitely needs "better buns" in this sense – fresh, soft, well-formed bread.\n\n2.  **"Buns" (Meaning 2 - Fitness Context):** In the context of going to the *gym*, "buns" is a common, informal term for the gluteal muscles (your buttocks). People go to the gym to "work on their buns" to make them stronger, firmer, or more shapely.\n\n**The Humor:**\n\nThe joke creates its humor by playing on this double meaning. You expect the punchline to relate to the burger\'s components, but it cleverly switches to the human/fitness meaning of "buns," 

In [ ]:
# Time travel (To resume from a specific checkpoint)

workflow.invoke(None,{'configurable':{'thread_id':thread_id,'checkpoint_id':'1f16ee00-cad4-6b04-8000-fb73cba17010'}})


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 18.479866549s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT